In [1]:
%load_ext autoreload
%autoreload 2

## Setup

In [68]:
import numpy as np
import pandas as pd

from principled_pclm.sim import SimGaussian1D
from principled_pclm.numpyro import IGMRF
from principled_pclm.models.smooth import make_bspline_basis
from principled_pclm.models import build_composition_matrix

import jax
import jax.numpy as jnp
from jax.random import PRNGKey

import numpyro
import numpyro.distributions as dist
from numpyro.contrib.hsgp.approximation import hsgp_matern, hsgp_squared_exponential
from principled_pclm.models._inference import run_inference_mcmc

# For plotting
import altair as alt

## Generate simulated data 

In [123]:
intervals = [
    "[0,10)",
    "[10,20)",
    "[20,30)",
    "[30,40)",
    "[40,50)",
    "[50,60)",
    "[60,70)",
    "[70,80)",
    "[80,90)",
    "[90,99]",
]

simulator = SimGaussian1D(
    obs_intervals=intervals,
    n_obs=10,
    omega_max=np.pi / 6,
    process_std=1,
    cutoff_ratio=0.3,
    damping_power=2,
    noise_std=0.1,
    seed=42,
)

df_train = simulator.generate()
df_ref = simulator.get_reference_df()

# Plot the true function
chart_base = alt.Chart(df_ref).mark_line().encode(x="x", y="f")

source_segment = df_ref[["interval_min", "interval_max", "mu"]].drop_duplicates()
chart_segment = (
    alt.Chart(source_segment)
    .mark_rule(color="orange", strokeWidth=2.5, opacity=0.85)
    .encode(
        x=alt.X("interval_min:Q", title="x"),
        x2="interval_max:Q",
        y=alt.Y("mu:Q", title="mu"),
    )
)

chart_base + chart_segment

alt.LayerChart(...)

## Build a PCLM model

In [124]:
# Build the composition matrix
C = build_composition_matrix(intervals)

# Build the B-spline basis
x = np.linspace(0, 99, 100)
Phi = jnp.array(make_bspline_basis(x, n_basis=40, natural_boundary=True))

# Observations
idx = jnp.array(df_train["idx"].values)
y = jnp.array(df_train["y"].values)

In [125]:
def pclm_model(C: jnp.ndarray, Phi: jnp.ndarray, idx: jnp.ndarray, y=None):
    """
    Bayesian B-spline regression model.

    Parameters
    ----------
    C: jnp.ndarray, shape (n_intervals, n_samples)
        Composition matrix.
    Phi : jnp.ndarray, shape (n_samples, n_basis)
        B-spline basis matrix.
    idx : jnp.ndarray, shape (n_samples,)
                Index of the intervals for each observation.
    """
    J, N = C.shape
    M = Phi.shape[1]  # number of basis functions

    # --- Priors ---
    tau = numpyro.sample("tau", dist.Gamma(2, 1))
    beta = numpyro.sample(
        "beta",
        IGMRF(num_nodes=M, cond_prec=tau, order=2, rescale_marginal_variance=True),
    )

    # --- Latent mean ---
    gamma = numpyro.deterministic("gamma", Phi @ beta)
    mu = numpyro.deterministic("mu", C @ gamma / C.sum(axis=1))

    # --- Likelihood ---
    with numpyro.plate("obs", N):
        numpyro.sample("y", dist.Normal(mu[idx], 0.1), obs=y)

In [135]:
# MCMC inference with NUTS
mcmc = run_inference_mcmc(PRNGKey(0), pclm_model, C=C, Phi=Phi, idx=idx, y=y)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/principled_pclm/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 2377.13it/s, 63 steps of size 8.88e-02. acc. prob=0.91]

Number of divergences: 0
dict_keys(['beta', 'gamma', 'mu', 'tau'])


In [136]:
# Extract posterior mean of mu from MCMC samples
gamma_central = mcmc_samples["gamma"].mean(axis=0)
gamma_lower = jnp.percentile(mcmc_samples["gamma"], 2.5, axis=0)
gamma_upper = jnp.percentile(mcmc_samples["gamma"], 97.5, axis=0)

df_post = df_ref.copy()
df_post["gamma_central"] = gamma_central
df_post["gamma_lower"] = gamma_lower
df_post["gamma_upper"] = gamma_upper

# Visualize results
true_line = alt.Chart(df_post).mark_line(color="red").encode(x="x", y="f")
mean_line = alt.Chart(df_post).mark_line(color="blue").encode(x="x", y="gamma_central")
post_interval = (
    alt.Chart(df_post)
    .mark_area(color="blue", opacity=0.2)
    .encode(x="x", y="gamma_lower", y2="gamma_upper")
)
true_line + chart_segment + mean_line + post_interval

alt.LayerChart(...)

In [137]:
df_post["within_ci"] = (df_post["f"] >= df_post["gamma_lower"]) & (
    df_post["f"] <= df_post["gamma_upper"]
)
print(df_post["within_ci"].mean())

0.86


## Principled GP based PCLM

In [128]:
from jax.typing import ArrayLike


class HSGP_PCLM:
    def __init__(
        self,
        x: ArrayLike,
        C: jnp.ndarray,
        idx: jnp.ndarray,
        length: float,
        ell: ArrayLike | None = None,
        m: int | None = None,
    ):
        """
        Parameters
        ----------
        x : jnp.ndarray
            Fine-scale grid locations at which the latent function is evaluated.
            Centered automatically before being passed to the HSGP approximation.
        C : jnp.ndarray
            Aggregation matrix mapping fine-scale rates to observed intervals.
        idx : jnp.ndarray
            Interval indices for each observation.
        length : float
            Fixed lengthscale of the Matern 3/2 kernel.
        ell : float | None, optional
            Boundary factor for the HSGP approximation.
        m : int | None, optional
            Number of basis functions used in the HSGP approximation.
        """
        self.x = jnp.asarray(x)
        self.xc = self.x - jnp.mean(self.x)
        self.C = C
        self.J, self.N = C.shape
        self.idx = idx
        self.length = length

        self.ell = (
            float(ell) if ell is not None else float(1.5 * jnp.max(jnp.abs(self.xc)))
        )
        self.m = (
            int(m)
            if m is not None
            else int(max(jnp.ceil(4.5 * self.ell / self.length), 10))
        )

    def model(self, y=None):
        W = self.C.sum(axis=1)

        # --- Priors ---
        alpha = numpyro.sample("alpha", dist.LogNormal(0, 1))

        # --- HSGP approximation of the latent GP (Matern nu=3/2, fixed lengthscale) ---
        f = hsgp_squared_exponential(
            x=self.xc,
            alpha=alpha,
            length=self.length,
            ell=self.ell,
            m=self.m,
            non_centered=True,
        )

        # --- Latent rate ---
        gamma = numpyro.deterministic("gamma", f)
        mu = self.C @ gamma / W

        # --- Likelihood ---
        with numpyro.plate("obs", len(y)):
            numpyro.sample("y", dist.Normal(mu[self.idx], 0.1), obs=y)

In [130]:
# Baseline model with naive lengthscale selection
hsgp_pcml_model = HSGP_PCLM(
    x=x, C=C, idx=idx, ell=2.0 * jnp.max(jnp.abs(x - jnp.mean(x))), length=6 * 0.85
)

In [131]:
mcmc = run_inference_mcmc(PRNGKey(0), hsgp_pcml_model.model, y=y)

/Users/shozen/Imperial/0_Research/mixres/principled_pclm/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:01<00:00, 894.42it/s, 127 steps of size 2.99e-02. acc. prob=0.91]

Number of divergences: 0


In [132]:
# Extract posterior mean of mu from MCMC samples
mcmc_samples = mcmc.get_samples()
gamma_central = mcmc_samples["gamma"].mean(axis=0)
gamma_lower = jnp.percentile(mcmc_samples["gamma"], 2.5, axis=0)
gamma_upper = jnp.percentile(mcmc_samples["gamma"], 97.5, axis=0)

df_post = df_ref.copy()
df_post["gamma_central"] = gamma_central
df_post["gamma_lower"] = gamma_lower
df_post["gamma_upper"] = gamma_upper

# Visualize results
true_line = alt.Chart(df_post).mark_line(color="red").encode(x="x", y="f")
mean_line = alt.Chart(df_post).mark_line(color="blue").encode(x="x", y="gamma_central")
post_interval = (
    alt.Chart(df_post)
    .mark_area(color="blue", opacity=0.2)
    .encode(x="x", y="gamma_lower", y2="gamma_upper")
)
true_line + chart_segment + mean_line + post_interval

alt.LayerChart(...)

In [133]:
df_post["within_ci"] = (df_post["f"] >= df_post["gamma_lower"]) & (
    df_post["f"] <= df_post["gamma_upper"]
)

print(df_post["within_ci"].mean())

0.97
